## 1.1 - Instalar librerias externas
First, search for the library, then download it and upload it to kaggle for using it without internet connection (Rules of the competition)

# 1-Librerias, rutas y datos CSV

In [ ]:
!pip install /kaggle/input/noisereduce/noisereduce-3.0.3-py3-none-any.whl

## 1.2 - Importar librerias

In [ ]:
# Librerias estandar
import os
import gc
import csv
import ast
import re
import json
import math
import shutil
import hashlib
import tarfile
import collections
import glob as gb
from pathlib import Path
from collections import Counter


# Manipulacion de datos
import numpy as np
import pandas as pd


# Audio y procesado de señal
import librosa
import librosa.display
import noisereduce as nr
from scipy.signal import butter, sosfiltfilt


# Visualizacion y progreso
import matplotlib.pyplot as plt
from tqdm import tqdm


# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# Modelos preentrenados
import timm
from torchvision import models as tv


# Scikit-learn
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold

## 1.3 - Crear rutas y directorios

### 1.3.1-Funciones

In [ ]:
def create_folder(path, carpeta):
    folder = os.path.join(path, carpeta)
    os.makedirs(folder, exist_ok=True)
    print(f'Folder "{carpeta}" created.')
    return folder

def get_path(dir, filename):
    return os.path.join(dir, filename)

def clean_folder(path, delete_folder=False):
    if delete_folder:
        shutil.rmtree(path)
        print(f"[+] Carpeta eliminada: {path}")
    else:
        for filename in os.listdir(path):
            file_path = os.path.join(path, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.remove(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f"[X] Error eliminando {file_path}: {e}")
        print(f"[+] Contenido eliminado en: {path}")

### 1.3.2-Creacion de rutas y directorios

In [ ]:
#  birdclef en /kaggle/input/birdclef-2025
path_birdclef = '/kaggle/input/birdclef-2025'
path_home = '/kaggle/working'
clean_folder(path_home)

path_models = create_folder(path_home, "models")

path_train_audio = get_path(path_birdclef, "train_audio")

path_taxonomy_csv = get_path(path_birdclef, "taxonomy.csv")
path_train_csv = get_path(path_birdclef, "train.csv")
path_sample_submission = get_path(path_birdclef, "sample_submission.csv")

path_data = create_folder(path_home, "data")
path_duplicates = create_folder(path_home, "duplicates")
path_suspects = create_folder(path_duplicates, "suspects")
path_duplicates_examples = create_folder(path_duplicates, "examples")
path_csvs = create_folder(path_data, "csv")

#path_prepmels = '/kaggle/input/prep-mels'

## 1.4-Cargar train_csv

In [ ]:
# Cargar taxonomy y train.csv
taxonomy = pd.read_csv(path_taxonomy_csv, encoding='ISO-8859-1')
train_df = pd.read_csv(path_train_csv, encoding='ISO-8859-1')

print(f"Ruta train.csv: {path_train_csv}")
print(train_df.head())

# Normalizar labels
taxonomy['primary_label'] = taxonomy['primary_label'].astype(str).str.strip()
train_df['primary_label'] = train_df['primary_label'].astype(str).str.strip()


# Filtrar solo especies válidas (las 206 oficiales)
sample = pd.read_csv(path_sample_submission)
CLASSES = list(sample.columns[1:])  # orden oficial
valid_labels = CLASSES
label_to_index = {lab: i for i, lab in enumerate(CLASSES)}
index_to_label = {i: lab for lab, i in label_to_index.items()}

train_df = train_df[train_df['primary_label'].isin(CLASSES)]
print(f"Audios filtrados por especie válida: {len(train_df)}")
#print(f"Especies taxonomy:{len(valid_labels)}")

print(len(label_to_index))

## 1.5-Dataset QA: Duplicate Audio & Label Conflicts Train CSV
Problema posterior encontrado, un audio bajo el mismo nombre, tamaño y duracion pero distinto hash en 4 grupos diferentes y con secondary label vacia. Analizando los audios descubri que son el mismo, es puede dar una posible fuga de datos en entrenamiento y validacion.

### 1.5.1-Deteccion de colisiones por basename en train.csv
La salida, se ve mas clara ejecutandolo desde una terminal. 

Analiza `train.csv` para detectar **colisiones por nombre de archivo**:

* Extrae `basename` de `filename` (el nombre final tipo `audio.ogg`).
* Encuentra basenames que aparecen **más de una vez** en el CSV.
* Para cada basename repetido resume:

  * cuántas filas lo contienen (`n_rows`)
  * cuántos `primary_label` distintos tiene (`n_labels`) y cuáles son
  * si `secondary_labels` está vacío o no (parseándolo bien a lista)
  * la unión de secundarios (`union_secondary`)
* Guarda un CSV resumen (`duplicates_groups.csv`) y, si hay conflicto (mismo basename con **distintos `primary_label`**), genera otro CSV con los **sospechosos** donde además `secondary_labels` está **efectivamente vacío** en todas las filas (casos más “raros” de posible mislabel/duplicado).

In [ ]:
# =========================
# CONFIG
# =========================
CSV_PATH = path_train_csv
OUT_GROUPS = get_path(path_duplicates, "duplicates_groups.csv")
OUT_SUSPECTS = get_path(path_duplicates, "duplicates_conflicts_secondary_label_empty.csv")
FLAG_DUPLICATES = False # True si se quiere buscar muestras de audio duplicadas y  sospechosas

# =========================
# HELPERS
# =========================
def safe_basename(p: str) -> str:
    return os.path.basename(str(p))

def parse_label_list(x):
    """
    Parsea campos como "['a','b']" o listas reales.
    Devuelve lista de strings LIMPIOS, eliminando vacíos ('', ' ', None).
    """
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    if isinstance(x, list):
        raw = x
    else:
        s = str(x).strip()
        if s == "" or s.lower() in {"nan", "none"}:
            return []
        # casos típicos de "vacío"
        if s in {"[]", "()", "{}"}:
            return []
        try:
            raw = ast.literal_eval(s)
        except Exception:
            # si no parsea, tratamos como un único token
            raw = [s]

    # Normaliza y filtra vacíos
    cleaned = []
    for item in raw:
        if item is None:
            continue
        t = str(item).strip()
        if t == "" or t.lower() in {"nan", "none"}:
            continue
        cleaned.append(t)
    return cleaned

# =========================
# MAIN
# =========================
if FLAG_DUPLICATES:
    # Load
    df = pd.read_csv(CSV_PATH)
    if "filename" not in df.columns:
        raise ValueError("No existe la columna 'filename' en el CSV")
    
    df["basename"] = df["filename"].apply(safe_basename)
    
    # Parse secondary_labels correctamente (si no existe, lo tratamos como vacío)
    if "secondary_labels" in df.columns:
        df["sec_list"] = df["secondary_labels"].apply(parse_label_list)
    else:
        df["sec_list"] = [[] for _ in range(len(df))]
    
    df["secondary_effectively_empty"] = df["sec_list"].apply(lambda lst: len(lst) == 0)
    
    # Repetidos
    counts = df["basename"].value_counts()
    rep_names = counts[counts > 1].index
    df_rep = df[df["basename"].isin(rep_names)].copy()
    
    print(f"Total filas: {len(df)}")
    print(f"Basenames repetidos (>1): {len(rep_names)}")
    print(f"Filas implicadas en repetidos: {len(df_rep)}")
    
    # Resumen por basename repetido
    agg = (
        df_rep.groupby("basename")
              .agg(
                  n_rows=("basename", "size"),
                  n_labels=("primary_label", pd.Series.nunique) if "primary_label" in df_rep.columns else ("basename", "size"),
                  labels=("primary_label", lambda s: sorted(set(map(str, s)))) if "primary_label" in df_rep.columns else ("basename", lambda s: []),
                  pct_secondary_empty=("secondary_effectively_empty", lambda s: float(s.mean())),
                  all_secondary_empty=("secondary_effectively_empty", lambda s: bool(s.all())),
                  union_secondary=("sec_list", lambda col: sorted(set([x for lst in col for x in lst]))),
              )
              .reset_index()
              .sort_values(["n_labels", "n_rows"], ascending=False)
    )
    
    agg.to_csv(OUT_GROUPS, index=False)
    print(f"[OK] {OUT_GROUPS}")
    
    # Conflictos con secondary efectivamente vacío (los “sospechosos” de mislabel/duplicado)
    if "primary_label" in df_rep.columns:
        conflicts = agg[agg["n_labels"] > 1].copy()
        suspects = conflicts[conflicts["all_secondary_empty"] == True].copy()
        print(f"Basenames con conflicto (n_labels>1): {len(conflicts)}")
        print(f"Conflictos donde secondary es efectivamente vacío (solo '', NaN, etc.): {len(suspects)}")
    
        # Guardar detalle fila-a-fila de esos sospechosos
        df_sus = df_rep[df_rep["basename"].isin(suspects["basename"])].copy()
        cols_first = [c for c in ["basename","primary_label","secondary_labels","sec_list","secondary_effectively_empty","filename","collection","rating","author"] if c in df_sus.columns]
        df_sus = df_sus[cols_first + [c for c in df_sus.columns if c not in cols_first]]
        df_sus.sort_values(["basename","primary_label"], inplace=True)
        df_sus.to_csv(OUT_SUSPECTS, index=False)
        print(f"[OK] {OUT_SUSPECTS}")
    
        if len(suspects) > 0:
            print("\n[+] Top 10 sospechosos:\n")
            print(suspects.head(10).to_string(index=False))
    else:
        print("[X] No existe 'primary_label' -> no se puede detectar conflicto de etiquetas.")

### 1.5.2-Verificacion de audios sospechosos
La salida se ve mas clara ejecutandolo desde una terminal
Genera un **reporte en CSV** a partir de tu lista de “sospechosos” (basenames) y **busca cada audio dentro de `train_audio/`** para:

* Guardar **la ruta relativa real** donde aparece en el dataset (formato `label/audio.ogg`, sin prefijo `train_audio/`).
* Indicar **cuántas copias** del mismo `basename` existen y **en cuántas carpetas/labels** diferentes aparece.
* (Opcional) Calcular el **hash MD5** de cada copia para saber si los archivos son **idénticos o distintos** aunque tengan el mismo

In [ ]:
# =========================
# CONFIG
# =========================
SUSPECTS_CSV = get_path(path_duplicates, "duplicates_conflicts_secondary_label_empty.csv")
TRAIN_AUDIO_ROOT = Path(path_train_audio)  
OUT_REPORT = get_path(path_suspects, "suspects_folder_check_report.csv")
OUT_SUMMARY = get_path(path_suspects, "suspects_folder_check_summary.csv")
OUT_MIXED = get_path(path_suspects, "suspects_folder_check_mixed_dirs.csv")

DO_HASH = True # True si quieres MD5 (más lento)

# =========================
# HELPERS
# =========================
def md5_file(path: Path, chunk_size: int = 1 << 20) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def parse_basenames_and_df(csv_path: str):
    df = pd.read_csv(csv_path)

    if "basename" not in df.columns:
        if "filename" in df.columns:
            df["basename"] = df["filename"].astype(str).apply(os.path.basename)
        else:
            raise ValueError("El CSV no tiene 'basename' ni 'filename'.")

    basenames = sorted(set(df["basename"].astype(str)))
    return basenames, df

def safe_rel(p: Path, root: Path) -> str:
    """Devuelve ruta relativa sin incluir el prefijo root (ej: label/file.ogg)."""
    try:
        return str(p.relative_to(root))
    except Exception:
        # fallback por si root no es padre (raro)
        return str(p)

# =========================
# MAIN
# =========================
if FLAG_DUPLICATES:
    if DO_HASH:
        print("[+] Flag obtener hash activado, este proceso puede ser mas lento")
    if not TRAIN_AUDIO_ROOT.exists():
        raise FileNotFoundError(f"No existe TRAIN_AUDIO_ROOT: {TRAIN_AUDIO_ROOT.resolve()}")
    
    basenames, df_sus = parse_basenames_and_df(SUSPECTS_CSV)
    
    # Mapeo basename -> lista de filename(s) del CSV (tipo grupo/nombre.ogg) SIN 'train_audio/'
    if "filename" in df_sus.columns:
        map_bname_to_csvfilenames = (
            df_sus.groupby("basename")["filename"]
                  .apply(lambda s: ";".join(sorted(set(map(str, s)))))
                  .to_dict()
        )
        map_bname_to_csvcount = (
            df_sus.groupby("basename")["filename"]
                  .apply(lambda s: len(set(map(str, s))))
                  .to_dict()
        )
    else:
        map_bname_to_csvfilenames = {}
        map_bname_to_csvcount = {}
    
    print(f"[+] Sospechosos (basenames) leídos: {len(basenames)}")
    print(f"[*] Buscando en: {TRAIN_AUDIO_ROOT.resolve()}")
    
    rows = []
    
    for bname in basenames:
        # Buscar todas las ocurrencias del fichero por nombre (recursivo)
        matches = list(TRAIN_AUDIO_ROOT.rglob(bname))
        found_copies_total = len(matches)
    
        # Si no hay copias encontradas
        if found_copies_total == 0:
            rows.append({
                "basename": bname,
                "csv_filenames": map_bname_to_csvfilenames.get(bname, ""),
                "csv_n_filenames": map_bname_to_csvcount.get(bname, 0),
                "found_copies": 0,
                "filename": "",
                "fs_label_dir": "",
                "dir_total_ogg": 0,
                "dir_only_this_file": False,
                "dir_has_other_ogg": False,
                "other_ogg_count": 0,
                "other_ogg_examples": "",
                "md5": "",
            })
            continue
    
        # Para cada copia encontrada en disco
        for p in matches:
            d = p.parent
    
            # Contar ogg dentro de esa carpeta/label
            oggs = sorted([x for x in d.glob("*.ogg")])
            other = [x.name for x in oggs if x.name != bname]
    
            fs_relpath = safe_rel(p, TRAIN_AUDIO_ROOT)            # ej: "1139490/CSA36385.ogg"
            fs_label_dir = safe_rel(d, TRAIN_AUDIO_ROOT)          # ej: "1139490" (o subcarpeta si la hubiera)
    
            rows.append({
                "basename": bname,
                "csv_filenames": map_bname_to_csvfilenames.get(bname, ""),
                "csv_n_filenames": map_bname_to_csvcount.get(bname, 0),
    
                "found_copies": found_copies_total,
                "filename": fs_relpath,
                "fs_label_dir": fs_label_dir,
    
                "dir_total_ogg": len(oggs),
                "dir_only_this_file": (len(oggs) == 1),
                "dir_has_other_ogg": (len(other) >= 1),
    
                "other_ogg_count": len(other),
                "other_ogg_examples": ";".join(other[:8]) if other else "",
                "md5": md5_file(p) if DO_HASH else "",
            })
    
    report = pd.DataFrame(rows)
    
    # Guardar reporte detallado
    report.to_csv(OUT_REPORT, index=False)
    print(f"[OK] Report detallado guardado: {OUT_REPORT}")
    shutil.copy(OUT_REPORT, get_path(path_csvs, "suspects_folder_check_report.csv"))
    
    # =========================
    # SUMMARY por basename
    # =========================
    def uniq_join(series):
        vals = sorted(set([str(x) for x in series if pd.notna(x) and str(x) != ""]))
        return ";".join(vals)
    
    summary_aggs = {
        "csv_n_filenames": ("csv_n_filenames", "max"),
        "csv_filenames": ("csv_filenames", "first"),
        "found_copies": ("found_copies", "max"),
        "unique_dirs": ("fs_label_dir", lambda s: s.nunique(dropna=True)),
        "dirs_only_this_file": ("dir_only_this_file", lambda s: int((s == True).sum())),
        "dirs_with_other_ogg": ("dir_has_other_ogg", lambda s: int((s == True).sum())),
        "any_dir_only_this_file": ("dir_only_this_file", lambda s: bool((s == True).any())),
        "any_dir_with_other_ogg": ("dir_has_other_ogg", lambda s: bool((s == True).any())),
        "fs_label_dirs": ("fs_label_dir", uniq_join),
    }
    
    if DO_HASH:
        summary_aggs["unique_md5"] = ("md5", lambda s: len(set([x for x in s if isinstance(x, str) and x != ""])))
    
    summary = (
        report.groupby("basename", as_index=False)
              .agg(**{k: pd.NamedAgg(column=v[0], aggfunc=v[1]) for k, v in summary_aggs.items()})
    )
    
    # Caso especial: aparece en 2+ dirs y hay mezcla (una dir con más ogg y otra con solo ese)
    summary["mixed_dirs_one_only_one_with_others"] = (
        (summary["unique_dirs"] >= 2) &
        (summary["any_dir_only_this_file"] == True) &
        (summary["any_dir_with_other_ogg"] == True)
    )
    
    # Orden útil
    summary = summary.sort_values(
        ["mixed_dirs_one_only_one_with_others", "found_copies", "unique_dirs"],
        ascending=[False, False, False]
    )
    
    summary.to_csv(OUT_SUMMARY, index=False)
    print(f"[OK] Resumen guardado: {OUT_SUMMARY}")
    
    # Guardar solo los "mixed"
    mixed = summary[summary["mixed_dirs_one_only_one_with_others"] == True].copy()
    
    
    # Prints rápidos
    print("\nResumen por basename (top 25):")
    cols_show = [
        "basename", "csv_n_filenames", "found_copies", "unique_dirs",
        "dirs_only_this_file", "dirs_with_other_ogg",
        "mixed_dirs_one_only_one_with_others"
    ]
    print(summary[cols_show].head(25).to_string(index=False))
    
    if len(mixed) > 0:
        mixed.to_csv(OUT_MIXED, index=False)
        print(f"[OK] Mixed dirs guardado: {OUT_MIXED}")
        print("\n[+] Basenames con el caso MIXED (una carpeta solo ese .ogg y otra con más .ogg):")
        print(mixed[cols_show + ["fs_label_dirs"]].head(30).to_string(index=False))
        print(f"[+] Para mas informacion abre el fichero {OUT_MIXED}")
    else:
        print("\n[+] No se encontraron casos MIXED con este input.")
    
    print(f"[+] Para mas informacion abre el fichero {OUT_REPORT}")

el siguiente script comprueba si los audios duplicados son identicos por su pcm hash o acustica

In [ ]:
# =========================
# CONFIG
# =========================
# 1) CSV de entrada:
#   - Puede ser tu summary que tiene: basename + csv_filenames (label/file.ogg;label2/file.ogg;...)
#   - O puede ser el report detallado que tiene: basename + fs_relpath (label/file.ogg)
IN_CSV = get_path(path_suspects, "suspects_folder_check_summary.csv")   # o "suspects_folder_check_report.csv" o "duplicates_conflicts_secondary_label_empty.csv"

# 2) Root real del audio:
TRAIN_AUDIO_ROOT = Path(path_train_audio)  # ajusta si lo tienes montado distinto

# 3) Parámetros de comparación acústica
SR = 32000
TRIM_SILENCE = True
TOP_DB = 30

N_MELS = 128
N_FFT = 2048
HOP = 512

# Umbrales (ajústalos si quieres ser más estricto)
THR_CORR = 0.995
THR_COS  = 0.999

# Outputs
OUT_PER_FILE = get_path(path_suspects, "suspects_audio_compare_per_file.csv")
OUT_PER_BASENAME = get_path(path_suspects, "suspects_audio_compare_per_basename.csv")

# =========================
# HELPERS
# =========================
def load_norm(path: Path, sr=SR, trim_silence=TRIM_SILENCE, top_db=TOP_DB):
    y, _ = librosa.load(path, sr=sr, mono=True)
    if trim_silence:
        y, _ = librosa.effects.trim(y, top_db=top_db)
    # z-norm
    y = y - float(np.mean(y))
    y = y / (float(np.std(y)) + 1e-8)
    return y

def pcm_hash(y: np.ndarray) -> str:
    """
    Hash del contenido decodificado (robusto frente a metadatos del ogg).
    Cuantizamos a int16 tras limitar rango.
    """
    yq = np.clip(y, -4, 4)
    yq = (yq / 4.0 * 32767.0).astype(np.int16)
    return hashlib.md5(yq.tobytes()).hexdigest()

def mel_mean_vec(y: np.ndarray, sr=SR, n_mels=N_MELS, n_fft=N_FFT, hop=HOP) -> np.ndarray:
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop)
    v = S.mean(axis=1).astype(np.float32)
    v = v / (np.linalg.norm(v) + 1e-8)
    return v

def corr_same_len(a: np.ndarray, b: np.ndarray) -> float:
    n = min(len(a), len(b))
    if n < 16:
        return np.nan
    a2 = a[:n]
    b2 = b[:n]
    return float(np.corrcoef(a2, b2)[0, 1])

def cos_sim(u: np.ndarray, v: np.ndarray) -> float:
    return float(np.dot(u, v) / ((np.linalg.norm(u) + 1e-8) * (np.linalg.norm(v) + 1e-8)))

def parse_paths_from_row(row) -> list[str]:
    """
    Intenta extraer lista de relpaths (label/file.ogg) desde columnas típicas:
      - csv_filenames: "a/b.ogg;c/d.ogg"
      - fs_label_dirs + basename (si existiese) -> no sirve sin nombre
      - fs_relpath en report (ya viene en filas separadas, no aquí)
      - filename (si viene con grupo/file.ogg)
    """
    if "csv_filenames" in row and isinstance(row["csv_filenames"], str) and row["csv_filenames"].strip():
        return [x.strip() for x in row["csv_filenames"].split(";") if x.strip()]
    if "filename" in row and isinstance(row["filename"], str) and row["filename"].strip():
        # podría ser una sola ruta por fila; devolver lista con 1
        return [row["filename"].strip()]
    return []

# =========================
# LOAD INPUT
# =========================
if FLAG_DUPLICATES:
    df = pd.read_csv(IN_CSV)
    
    if "basename" not in df.columns:
        if "filename" in df.columns:
            df["basename"] = df["filename"].astype(str).apply(os.path.basename)
        else:
            raise ValueError("El CSV de entrada no tiene 'basename' ni 'filename'.")
    
    if not TRAIN_AUDIO_ROOT.exists():
        raise FileNotFoundError(f"TRAIN_AUDIO_ROOT no existe: {TRAIN_AUDIO_ROOT}")
    
    # =========================
    # BUILD groups: basename -> list of relpaths
    # =========================
    groups = {}
    
    if "fs_relpath" in df.columns:
        # Modo report: una fila por copia (basename + fs_relpath)
        for b, sub in df.groupby("basename"):
            rels = sorted(set([str(x) for x in sub["fs_relpath"].dropna().tolist() if str(x).strip()]))
            if rels:
                groups[b] = rels
    
    else:
        # Modo summary o suspects: una fila por basename con csv_filenames
        for _, row in df.iterrows():
            b = str(row["basename"])
            rels = parse_paths_from_row(row)
            if rels:
                groups[b] = sorted(set(rels))
    
    # Si tu IN_CSV fuera el de "duplicates_conflicts_secondary_label_empty.csv"
    # (fila-a-fila), esto agrupa por basename y recoge todos los filename:
    if not groups and "filename" in df.columns:
        for b, sub in df.groupby("basename"):
            rels = sorted(set([str(x) for x in sub["filename"].dropna().tolist() if str(x).strip()]))
            if rels:
                groups[b] = rels
    
    print(f"[OK] basenames a comprobar: {len(groups)}")
    
    # =========================
    # PROCESS
    # =========================
    per_file_rows = []
    per_base_rows = []
    
    for basename, relpaths in groups.items():
        # Construir paths reales
        abs_paths = [(rp, TRAIN_AUDIO_ROOT / rp) for rp in relpaths]
    
        # Cargar todas (con control de errores)
        items = []
        for rp, ap in abs_paths:
            if not ap.exists():
                per_file_rows.append({
                    "basename": basename,
                    "relpath": rp,
                    "exists": False,
                    "error": "missing_file",
                    "pcm_hash": "",
                    "wave_len": 0,
                    "corr_to_anchor": np.nan,
                    "cos_melmean_to_anchor": np.nan,
                })
                continue
    
            try:
                y = load_norm(ap)
                h = pcm_hash(y)
                mv = mel_mean_vec(y)
                items.append((rp, ap, y, h, mv))
                per_file_rows.append({
                    "basename": basename,
                    "relpath": rp,
                    "exists": True,
                    "error": "",
                    "pcm_hash": h,
                    "wave_len": int(len(y)),
                    "corr_to_anchor": np.nan,          # se rellena después
                    "cos_melmean_to_anchor": np.nan,   # se rellena después
                })
            except Exception as e:
                per_file_rows.append({
                    "basename": basename,
                    "relpath": rp,
                    "exists": False,
                    "error": f"load_error:{type(e).__name__}",
                    "pcm_hash": "",
                    "wave_len": 0,
                    "corr_to_anchor": np.nan,
                    "cos_melmean_to_anchor": np.nan,
                })
    
        # Si no hay items válidos
        if len(items) == 0:
            per_base_rows.append({
                "basename": basename,
                "n_relpaths": len(relpaths),
                "n_loaded": 0,
                "n_missing_or_error": len(relpaths),
                "unique_pcm_hash": 0,
                "min_corr_to_anchor": np.nan,
                "min_cos_to_anchor": np.nan,
                "all_same_pcmhash": False,
                "all_acoustically_same": False,
                "status": "NO_FILES_LOADED",
            })
            continue
    
        # Anchor = primer item válido
        anchor_rp, anchor_ap, yA, hA, mvA = items[0]
    
        corrs = []
        coss = []
        hashes = set([h for _,_,_,h,_ in items])
    
        # Actualizar métricas a anchor en per_file_rows (las filas ya creadas)
        # Haremos un mapping rápido
        metrics_map = {anchor_rp: (1.0, 1.0)}
        for rp, ap, y, h, mv in items[1:]:
            c = corr_same_len(yA, y)
            cs = cos_sim(mvA, mv)
            metrics_map[rp] = (c, cs)
            corrs.append(c)
            coss.append(cs)
    
        # corr/cos de anchor con sí mismo
        if anchor_rp not in metrics_map:
            metrics_map[anchor_rp] = (1.0, 1.0)
    
        # Parche: si solo hay 1 fichero, min = 1
        min_corr = float(np.nanmin(corrs)) if len(corrs) > 0 else 1.0
        min_cos  = float(np.nanmin(coss))  if len(coss) > 0 else 1.0
    
        all_same_pcm = (len(hashes) == 1)
        all_acoustic = (min_corr >= THR_CORR) and (min_cos >= THR_COS)
    
        if all_same_pcm:
            status = "EXACT_PCM_IDENTICAL"
        elif all_acoustic:
            status = "ACOUSTICALLY_VERY_SIMILAR"
        else:
            status = "DIFFERENT_OR_MISMATCH"
    
        # Actualiza filas per_file con corr/cos al anchor
        # (solo para los que existían/cargaron)
        for r in per_file_rows:
            if r["basename"] == basename and r["exists"] == True and r["relpath"] in metrics_map:
                c, cs = metrics_map[r["relpath"]]
                r["corr_to_anchor"] = c
                r["cos_melmean_to_anchor"] = cs
    
        per_base_rows.append({
            "basename": basename,
            "anchor_relpath": anchor_rp,
            "n_relpaths": len(relpaths),
            "n_loaded": len(items),
            "n_missing_or_error": len(relpaths) - len(items),
            "unique_pcm_hash": len(hashes),
            "min_corr_to_anchor": min_corr,
            "min_cos_to_anchor": min_cos,
            "all_same_pcmhash": all_same_pcm,
            "all_acoustically_same": all_acoustic,
            "status": status,
        })
    
    # =========================
    # SAVE OUTPUTS
    # =========================
    df_files = pd.DataFrame(per_file_rows)
    df_base  = pd.DataFrame(per_base_rows).sort_values(
        ["status", "unique_pcm_hash", "min_corr_to_anchor", "min_cos_to_anchor"],
        ascending=[True, True, True, True]
    )
    
    df_files.to_csv(OUT_PER_FILE, index=False)
    df_base.to_csv(OUT_PER_BASENAME, index=False)
    
    print(f"[OK] Guardado: {OUT_PER_FILE}")
    print(f"[OK] Guardado: {OUT_PER_BASENAME}")
    
    #print("\n[+] Top 10 basenames más 'sospechosos' (no iguales):")
    #print(df_base[df_base["status"] == "DIFFERENT_OR_MISMATCH"].head(10).to_string(index=False))

### 1.5.2-Mostrar la onda de dos muestras "iguales"

In [ ]:
# =========================
# CONFIG
# =========================

PATH_A = get_path(path_train_audio, "blcant4/iNat1020465.ogg")   
PATH_B = get_path(path_train_audio,"greibi1/iNat1020465.ogg")
PATH_C = get_path(path_train_audio,"bubwre1/iNat1020464.ogg") # muestra diferente para comprobar funcionamiento
name_a = "blcant4_iNat1020465"
name_b ="greibi1_iNat1020465"
name_c ="bubwre1_iNat1020464"
SR = 32000

# =========================
# HELPERS
# =========================
def get_audio(path, sr=32000):
    y, sr = librosa.load(path, sr=sr, mono=True)
    return y, sr

def plot_compare(a_path, b_path, a_name="A", b_name="B", sr=32000, sec=12.0):
    label_a = a_name.split("_")[0]
    label_b = b_name.split("_")[0]

    ya, _ = get_audio(a_path, sr=sr)
    yb, _ = get_audio(b_path, sr=sr)

    n = int(sec * sr)
    ya = ya[:n]
    yb = yb[:n]

    t = np.arange(len(ya)) / sr

    # superpuesto
    plt.figure(figsize=(14, 4))
    plt.plot(t, ya, label=f"{a_name}: {Path(a_path).parent.name}/{Path(a_path).name}", alpha=0.8)
    plt.plot(t, yb, label=f"{b_name}: {Path(b_path).parent.name}/{Path(b_path).name}", alpha=0.8)
    plt.title("Waveform overlay (primeros segundos)")
    plt.xlabel("Tiempo (s)")
    plt.legend()
    plt.tight_layout()
    save_name = get_path(path_duplicates_examples, f"superpuesto_{label_a}_{label_b}.png")
    plt.savefig(save_name, dpi=300)
    plt.show()

    
    # diferencia, mirar eje izq, si salen valores cercanos al 0 son idenaticos, si salen 0,1;0,2;0,3 no lo son
    diff = ya - yb
    t = np.arange(n) / SR
    
    plt.figure(figsize=(14,3))
    plt.plot(t, diff)
    plt.title(f"Diferencia ({label_a} - {label_b}). Si es ~0, son casi idénticas")
    plt.tight_layout()
    save_name = get_path(path_duplicates_examples, f"diferencias_{label_a}_{label_b}.png")
    plt.savefig(save_name, dpi=300)
    plt.show()

    # Espectrograma mel A
    for y, title in [(ya, a_name), (yb, b_name)]:
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=512, n_mels=128, fmin=20, fmax=16000)
        S_db = librosa.power_to_db(S, ref=np.max)
        plt.figure(figsize=(14, 4))
        librosa.display.specshow(S_db, sr=sr, hop_length=512, x_axis="time", y_axis="mel")
        plt.colorbar(format="%+2.0f dB")
        plt.title(f"Mel-spectrogram {title} ({sec}s)")
        plt.tight_layout()
        save_name = get_path(path_duplicates_examples, f"mel_{title}.png")
        plt.savefig(save_name, dpi=300)
        plt.show()
        plt.close()

if FLAG_DUPLICATES:
    print(f"[*] Comprobando dos muestras con nombre: {name_a} y {name_b}\n")
    plot_compare(PATH_A, PATH_B,a_name=name_a, b_name=name_b, sec=12.0)
    print(f"\n\n[*] Comprobando dos muestras con nombre: {name_a} y {name_c}\n")
    plot_compare(PATH_A, PATH_C,a_name=name_a, b_name=name_c, sec=12.0)

### 1.5.3-Mostrar su correlacion
Si los valores estan lo mas cercano a 1, se puede afirmar que las muestras son identicas

In [ ]:
if FLAG_DUPLICATES:
    ya, _ = librosa.load(PATH_A, sr=SR, mono=True)
    yb, _ = librosa.load(PATH_B, sr=SR, mono=True)
    yc, _ = librosa.load(PATH_C, sr=SR, mono=True)
        
    print(f"[*] Comprobando dos muestras con nombre: {name_a} y {name_b}\n")
    # COMPROBAR MUESTRAS IDENTICAS
    # igualar longitud
    n = min(len(ya), len(yb))
    ya = ya[:n]
    yb = yb[:n]
    
    # correlación normalizada
    corr = np.corrcoef(ya, yb)[0,1]
    print("[+] Corr(waveform):", corr)
    
    # similitud en mel promedio (más robusto a desfases)
    Sa = librosa.feature.melspectrogram(y=ya, sr=SR, n_mels=128, n_fft=2048, hop_length=512)
    Sb = librosa.feature.melspectrogram(y=yb, sr=SR, n_mels=128, n_fft=2048, hop_length=512)
    ma = Sa.mean(axis=1); mb = Sb.mean(axis=1)
    cos = float(np.dot(ma, mb) / (np.linalg.norm(ma)*np.linalg.norm(mb) + 1e-9))
    print("[+] Cosine(mel-mean):", cos)
    
    
    print(f"\n\n[*] Comprobando dos muestras con nombre: {name_a} y {name_c}\n")
    # COMPROBAR MUESTRAS DIFERENTES
    # igualar longitud
    n = min(len(ya), len(yc))
    ya = ya[:n]
    yc = yc[:n]
    
    # correlación normalizada
    corr = np.corrcoef(ya, yc)[0,1]
    print("[+] Corr(waveform):", corr)
    
    # similitud en mel promedio (más robusto a desfases)
    Sa = librosa.feature.melspectrogram(y=ya, sr=SR, n_mels=128, n_fft=2048, hop_length=512)
    Sc = librosa.feature.melspectrogram(y=yc, sr=SR, n_mels=128, n_fft=2048, hop_length=512)
    ma = Sa.mean(axis=1); mc = Sc.mean(axis=1)
    cos = float(np.dot(ma, mc) / (np.linalg.norm(ma)*np.linalg.norm(mc) + 1e-9))
    print("[+] Cosine(mel-mean):", cos)

# 2 - Preprocesamiento de audio

## 2.0-Configuracion global - Parametros

In [ ]:
### Parametros modificables ###
# Audio / STFT Config
SAMPLE_RATE = 32000 # Tasa de muestreo, formato del dataset
N_FFT = 1024 # Transformada de Fourier
HOP_LENGTH = 320 # nº muestras entre el inicio de ventanas, ver stft
F_MIN = 20 # Frecuencia minima Hz
F_MAX = 16000 # Frecuencia maxima Hz
MIN_SEG_DURATION_SEC = 0.5 # Duracion minima para intervalos

# Mels
N_MELS = 128 # Nº de bandas del espectograma de mel
MEL_DURATION = 5.0  # Duracion en segundos por segmento

# VAD Recortes
TOP_DB = None #30# Umbral para detectar actividad, <30 = silencio

# noise reduce
NOISE_REDUCE = False

# Filtros de banda
APPLY_BANDPASS = True # True si se quiere aplicar, False en caso contrario
LOWCUT = 150 #300  # Frecuencia de corte inferior
HIGHCUT = 15550 #9000 #15950  # Frecuencia de corte superior
ORDER = 5 # Orden del filtro, 5 valor intermedio

# Speg Aumentation
APPLY_SPECAUGMENT = True # True, aplicar data augmentation al espectograma
TIME_MASK= 20 #40 # ocultar columnas aletorias
FREQ_MASK= 8 #12 # ocultar filas aleatorias
PROB = 0.5 # 0.4 # probabilidad

# Pipelina Preprocesado
FLAG_PROCESS= False # True si se quiere preprocesar los datos de validacion
FLAG_MAP = False # True si se quiere mapear el train csv
FLAG_LOAD_CSV = True # True si se quiere cargar desde un dataset externo index train, val train, index audio
FLAG_LOAD_SUSPECTS = True # True si se quiere cargar el csv de un input de kaggle
FLAG_OMIT_SUSPECTS = True # True si se quiere omitir audios duplicados en el mapeo, solo aplica a flag map si esta activado
FLAG_MAKE_TEST = True # True para separar una part del conjunto validacion para un test propio
###
### FLAG DUPLICATES APARTADO 1.5 ###
###

RNG_SEED = 42 # semilla para muestreo

## 2.1 - Funciones de limpieza y preprocesado
Includes noise reduce, bandpass filter, normalize audio, detect active segments,

### 2.1.1-Preprocesado base:

In [ ]:
# Funciones

def load_audio(filepath, sr=SAMPLE_RATE):
    # Cargar el archivo de audio
    y, sr = librosa.load(filepath, sr=sr, mono=True, dtype=np.float32, res_type='kaiser_fast')
    return y, sr


def reduce_noise(y, sr):
    # Reducir el ruido
    y_denoised = nr.reduce_noise(y=y, sr=sr)
    return y_denoised


def bandpass_filter(y, sr, lowcut=LOWCUT, highcut=HIGHCUT, order=ORDER):
    if not APPLY_BANDPASS:
        return y
    # Filtrado para eliminar frecuencias irrelevantes (Filtro paso bajo y filtro paso alto)
    nyquist = 0.5 * sr
    low = max(1.0, float(lowcut))
    high = min(float(highcut), nyquist * 0.98)
    if low >= high: # datos invalidos
        return y
    low = lowcut / nyquist
    high = highcut / nyquist
    # Butterworth formato sos y filtrado cero-fase
    sos= butter(order, [low, high], btype='band', output='sos')
    y_filtered = sosfiltfilt(sos, y) # evitar desfase
    return y_filtered


def normalize_audio(y):
    # Normalizacion la señal del audio, par aque todos los audios tengan un volumen similar
    # (ya sea entre -1 y 1)
    y_normalized = librosa.util.normalize(y)
    return y_normalized


def detect_active_segments(y, sr, top_db=TOP_DB):
    if top_db is None:
        return np.array([[0, len(y)]], dtype=int)
    # devuelve array de pares [start, end]
    return librosa.effects.split(y, top_db=top_db)


def preprocess_wave(y, sr, use_bp=False, do_norm=True, denoise_fn=False):
    if denoise_fn:
        y = denoise_fn(y=y, sr=sr)
    if use_bp:
        y = bandpass_filter(y, sr)
    if do_norm:
        y = normalize_audio(y)
    return y

### 2.1.2-Funcion Spec Augmentation:

In [ ]:
def spec_augment(mel, time_mask=TIME_MASK, freq_mask=FREQ_MASK, p=PROB):
    """
    Enmascaramiento temporal y frecuencial ligero y probabilístico.
    - Evita tamaños 0 o mayores que la dimensión.
    - No siempre aplica (p) para no destruir la señal sistemáticamente.
    """
    M = mel.copy()
    n_mels, n_frames = M.shape

    if np.random.rand() < p and n_frames > 1 and time_mask > 0:
        t = np.random.randint(1, min(time_mask, n_frames))
        t0 = np.random.randint(0, n_frames - t + 1)
        M[:, t0:t0+t] = 0

    if np.random.rand() < p and n_mels > 1 and freq_mask > 0:
        f = np.random.randint(1, min(freq_mask, n_mels))
        f0 = np.random.randint(0, n_mels - f + 1)
        M[f0:f0+f, :] = 0

    return M

### 2.1.3-Extracción de mels

In [ ]:
def extract_mel(y_segment, sr,
                n_mels=N_MELS,
                hop_length=HOP_LENGTH,
                duration=MEL_DURATION,
                n_fft=N_FFT,
                fmin=F_MIN,
                fmax=F_MAX,
                apply_specaug=APPLY_SPECAUGMENT):
    """
    - Ajusta el segmento a 'duration' segundos (pad/cut).
    - Calcula Mel con tus hiperparámetros globales.
    - Usa log1p(mel) para estabilidad numérica y mejor comportamiento en training.
    - Devuelve [n_mels, n_frames] en float32.
    """
    # Ajuste exacto de longitud
    samples = int(duration * sr)
    y_fixed = librosa.util.fix_length(y_segment, size=samples)

    # Mel spectrogram (power=2.0 → espectro de potencia)
    mel = librosa.feature.melspectrogram(
        y=y_fixed,
        sr=sr,
        n_mels=n_mels,
        hop_length=hop_length,
        n_fft=n_fft,
        fmin=fmin,
        fmax=fmax,
        power=2.0
    )

    # Log-mel estable
    mel = np.log1p(mel).astype(np.float32)
    #mel = (mel - mel.mean()) / (mel.std() + 1e-6)

    # SpecAugment (solo entrenamiento)
    if apply_specaug:
        mel = spec_augment(mel)

    return mel

# 3 - Dataset & Dataloader

## 3.0- Obtener duplicados

In [ ]:
if FLAG_LOAD_SUSPECTS:
    origen = "/kaggle/input/prep-mels/suspects_folder_check_report.csv"
    destino = get_path(path_suspects, "suspects_folder_check_report.csv")
    shutil.copy(origen, destino)
    print(f"[+] Copiado: {destino}")
            
REPORT_CSV = get_path(path_suspects, "suspects_folder_check_report.csv")

if FLAG_OMIT_SUSPECTS:
    rep = pd.read_csv(REPORT_CSV)
    not_allowed = set(rep["filename"].astype(str).tolist())
    print("[+] not_allowed size:", len(not_allowed))
    print("[+] Ejemplos:", list(sorted(not_allowed))[:10])

## 3.1.-Mapeo del dataset

In [ ]:
# ----------------------------
# Parámetros de mapeo
# ----------------------------
WIN_S         = 5.0        # duración de la ventana base
STRIDE_S      = 5.0        # sin solape para entrenamiento (puedes cambiar a 2.5 en inferencia)
MIN_DURATION  = 1.0        # ignora audios < 1s

# Entradas esperadas
# - train_df: DataFrame original (con 'filename', 'primary_label', ... )
# - path_train_audio: carpeta raíz del audio (función get_path ya la tienes)
# Si no tienes get_path, usa: os.path.join(path_train_audio, row['filename'])

np.random.seed(RNG_SEED)

# ----------------------------
# Helpers
# ----------------------------
def safe_get_duration(path, sr=SAMPLE_RATE):
    """Duración en segundos; devuelve None si falla."""
    try:
        return librosa.get_duration(path=path, sr=sr)
    except Exception as e:
        return None

def compute_starts(duration, win_s=WIN_S, stride_s=STRIDE_S):
    """
    Genera posiciones de inicio (en segundos) para ventanas de longitud win_s.
    Sin solape para entrenamiento: stride = win_s (aquí configurable).
    Garantiza al menos una ventana si el audio >= MIN_DURATION.
    """
    if duration is None or duration < MIN_DURATION:
        return []
    if duration < win_s:
        # Si el clip es corto, aún así genera un único inicio 0.0 (se hará pad en extract_mel)
        return [0.0]
    n = int(math.floor((duration - win_s) / stride_s)) + 1
    return [round(i * stride_s, 3) for i in range(n)]

# ----------------------------
# Mapeo principal
# ----------------------------
if FLAG_MAP:
    index_rows = []       # 1 fila por clip
    index_long_rows = []  # 1 fila por ventana (opcional)
    
    print("[+] Construyendo índice de audio...")
    for i, row in tqdm(train_df.iterrows(), total=len(train_df)):
        filename = row["filename"]

        # omitir el audio si es uno de los duplicados y esta activada esta opcion
        if FLAG_OMIT_SUSPECTS and filename in not_allowed:
            continue
        
        label    = row.get("primary_label", None)
        author   = row.get("author", "")
        site     = row.get("site", "") if "site" in row else row.get("location", "")
        path     = get_path(path_train_audio, filename)  # ajusta si no usas get_path
    
        if not os.path.exists(path):
            # intenta ruta alternativa si tu estructura tiene subcarpetas por especie
            alt_path = os.path.join(path_train_audio, filename)
            path = alt_path
    
        dur = safe_get_duration(path, sr=SAMPLE_RATE)
        starts = compute_starts(dur, win_s=WIN_S, stride_s=STRIDE_S)
    
        # Índice por clip (compacto)
        index_rows.append({
            "clip_id": i,
            "filename": filename,
            "path": path,
            "primary_label": label,
            "author": author,
            "site": site,
            "duration_sec": None if dur is None else round(dur, 3),
            "n_windows": len(starts),
            "starts_json": json.dumps(starts)  # lista compacta
        })
    
        # Índice largo (una fila por ventana)
        for s in starts:
            index_long_rows.append({
                "clip_id": i,
                "filename": filename,
                "path": path,
                "primary_label": label,
                "start_sec": s,
                "win_sec": WIN_S
            })
    
    # ----------------------------
    # Guardado
    # ----------------------------
    index_df = pd.DataFrame(index_rows)
    index_long_df = pd.DataFrame(index_long_rows)
    
    out_dir = path_csvs  # "./"
    index_csv_path = os.path.join(out_dir, "index_audio.csv")
    index_long_csv_path = os.path.join(out_dir, "index_audio_long.csv")
    
    index_df.to_csv(index_csv_path, index=False)
    index_long_df.to_csv(index_long_csv_path, index=False)
    
    print(f"[+] Guardado índice por clip: {index_csv_path} (filas={len(index_df)})")
    print(f"[+] Guardado índice por ventana: {index_long_csv_path} (filas={len(index_long_df)})")
    
    # liberar memoria
    del index_rows, index_long_rows
    gc.collect()
else:
    print("[+] Mapeo desactivado")

## 3.2-DataLoader

### 3.2.1-Clase OnThe Fly

In [ ]:
class OnTheFlyWindows(Dataset):
    def __init__(self, index_csv, label_map, split="train",
                 window_s=5.0, sr=32000, hop_policy=("random" if True else "fixed"),
                 bandpass=False, denoise_fn=False, specaug=False, time_mask=40, freq_mask=12, prob=0.5):
        self.df = pd.read_csv(index_csv)  # usa tu index_audio.csv (por clip) o index_audio_long.csv (por ventana)
        self.label_map = label_map
        self.split = split
        self.window = int(window_s*sr)
        self.sr = sr
        self.hop_policy = hop_policy
        self.apply_bandpass = bandpass
        self.denoise_fn = denoise_fn
        self.specaug = specaug
        self.time_mask = time_mask
        self.freq_mask = freq_mask
        self.prob = prob

    def __len__(self): return len(self.df)

    def _specaug(self, mel, time_mask=40, freq_mask=12, p=0.5):
        """
        Enmascaramiento temporal y frecuencial ligero y probabilístico.
        - Evita tamaños 0 o mayores que la dimensión.
        - No siempre aplica (p) para no destruir la señal sistemáticamente.
        """
        M = mel.copy()
        n_mels, n_frames = M.shape
        
        if np.random.rand() < p and n_frames > 1 and time_mask > 0:
            t = np.random.randint(1, min(time_mask, n_frames))
            t0 = np.random.randint(0, n_frames - t + 1)
            M[:, t0:t0+t] = 0
        
        if np.random.rand() < p and n_mels > 1 and freq_mask > 0:
            f = np.random.randint(1, min(freq_mask, n_mels))
            f0 = np.random.randint(0, n_mels - f + 1)
            M[f0:f0+f, :] = 0
        
        return M
        
    def _extract_mel(self, y_segment, sr,
                n_mels=N_MELS, hop_length=HOP_LENGTH,
                duration=MEL_DURATION, n_fft=N_FFT,
                fmin=F_MIN, fmax=F_MAX):
        """
        - Ajusta el segmento a 'duration' segundos (pad/cut).
        - Calcula Mel con tus hiperparámetros globales.
        - Usa log1p(mel) para estabilidad numérica y mejor comportamiento en training.
        - Devuelve [n_mels, n_frames] en float32.
        """
        # Ajuste exacto de longitud
        samples = int(duration * sr)
        y_fixed = librosa.util.fix_length(y_segment, size=samples)
    
        # Mel spectrogram (power=2.0 → espectro de potencia)
        mel = librosa.feature.melspectrogram(
            y=y_fixed,
            sr=sr,
            n_mels=n_mels,
            hop_length=hop_length,
            n_fft=n_fft,
            fmin=fmin,
            fmax=fmax,
            power=2.0
        )
    
        # Log-mel estable
        mel = np.log1p(mel).astype(np.float32)
        #mel = (mel - mel.mean()) / (mel.std() + 1e-6)

        return mel

    def __getitem__(self, i):
        r = self.df.iloc[i]
        path = r["path"] if "path" in r else r["filename"]
        label = self.label_map.get(r["primary_label"], -1)

        y, sr = librosa.load(path, sr=self.sr, mono=True)
        # elegir inicio (train aleatorio; val/test fijo)
        if self.split == "train":
            if len(y) > self.window:
                start = np.random.randint(0, len(y)-self.window)
            else:
                start = 0
        else:
            start = int(r["start"]*sr) if "start" in r else 0

        # recorte/padding
        seg = y[start:start+self.window]
        if len(seg) < self.window:
            seg = np.pad(seg, (0, self.window-len(seg)))

        # preproc on-the-fly
        seg = preprocess_wave(seg, self.sr, use_bp=self.apply_bandpass, do_norm=True, denoise_fn=self.denoise_fn)
        mel = self._extract_mel(seg, self.sr) # ya se aplica despues spec aug
        
        # augment solo en train
        if self.split == "train" and self.specaug:
            m = self._specaug(mel, self.time_mask, self.freq_mask, p=self.prob)
            mel = m

        # z-score por clip
        m = (mel - mel.mean()) / (mel.std()+1e-6)
        #m = mel

        return {"mel": m, "label": label, "path": path, "start": start/sr}

### 3.2.2-DataLoader Validacion

In [ ]:
class ValGroupedDataset(Dataset):
    def __init__(self, groups, index_csv, label_to_index, num_classes=206):
        self.audios = sorted(groups.keys())     # paths absolutos
        self.groups = groups
        self.num_classes = num_classes

        df = pd.read_csv(index_csv)
        df["filename"] = df["filename"].astype(str).str.strip()
        df["primary_label"] = df["primary_label"].astype(str).str.strip()
        df["primary_label"] = df["primary_label"].str.replace(r"\.0$", "", regex=True)

        # map robusto con aliases
        mapped = df["primary_label"].apply(lambda x: label_to_index.get(str(x).strip(), -1)).astype(int)
        self.file2label = dict(zip(df["filename"], mapped))

    def __len__(self):
        return len(self.audios)

    def _path_to_filename(self, path):
        marker = "/train_audio/"
        if marker in path:
            return path.split(marker, 1)[1]  # "grupo/audio.ogg"
        return os.path.basename(path)

    def __getitem__(self, i):
        audio_path = self.audios[i]
        filename = self._path_to_filename(audio_path)

        lid = self.file2label.get(filename, -1)

        y = torch.zeros(self.num_classes)
        if lid >= 0:
            y[lid] = 1.0

        return self.groups[audio_path], y, audio_path

## 3.3-Precomputar muestras validacion

In [ ]:
VAL_INDEX = get_path(path_csvs, "index_val.csv")
OUT_DIR = "./data/val_npz"     # npz temporales
ARCH_DIR = "./npz"            # .tar.gz finales
STRIDE_S = 2.5

def archive_current_chunk(part_idx, src_dir=OUT_DIR, dst_dir=ARCH_DIR):
    os.makedirs(dst_dir, exist_ok=True)
    tar_path = os.path.join(dst_dir, f"val_npz_part{part_idx:02d}.tar.gz")
    with tarfile.open(tar_path, "w:gz") as tar:
        tar.add(src_dir, arcname=os.path.basename(src_dir))
    print(f"[+] Checkpoint {part_idx:02d} → {tar_path}")

def clean_npz_dir(path=OUT_DIR):
    # Usa tu clean_folder si la tienes:
    try:
        print(f"[*] Eliminando contenido en {path} ...")
        clean_folder(path)  # tu utilidad
    except NameError:
        # fallback seguro
        for root, dirs, files in os.walk(path):
            for f in files:
                os.remove(os.path.join(root, f))

def precompute_npz_chunked(index_csv=VAL_INDEX, out_dir=OUT_DIR,
                           sr=SAMPLE_RATE, window_s=MEL_DURATION, stride_s=STRIDE_S, bandpass=APPLY_BANDPASS):
    os.makedirs(out_dir, exist_ok=True)
    df = pd.read_csv(index_csv)
    n = len(df)
    print(f"[+] Archivos a procesar: {n}")

    # thresholds 10%,20%,...,100%
    thresholds = {int(n * p / 100): i for i, p in enumerate(range(10, 101, 10), start=1)}
    part_idx = 0
    processed = 0

    print(f"[+] Bandpass filter status: {bandpass}")
    
    for _, r in tqdm(df.iterrows(), total=n, desc="Precomputando NPZ", ncols=100):
        path = r["path"] if "path" in r else r["filename"]
        y, _ = librosa.load(path, sr=sr, mono=True)
        y = preprocess_wave(y, sr, use_bp=bandpass, do_norm=True, denoise_fn=False)

        win = int(window_s * sr); hop = int(stride_s * sr)
        starts = range(0, max(1, len(y) - win + 1), hop)

        # para evitar colision de datos, ficheros con el mismo nombre, distinto grupo
        filename = str(r.get("filename", path))   # fallback a path si no hay filename
        clip_id = os.path.splitext(filename)[0]   # quita .ogg
        clip_id = clip_id.replace("/", "__") 
        
        for s in starts:
            seg = y[s:s+win]
            if len(seg) < win:
                seg = np.pad(seg, (0, win - len(seg)))

            mel = extract_mel(seg, sr, apply_specaug=False) # en validacion no se aplica spec augmentation
            # z-score
            mel = (mel - mel.mean()) / (mel.std() + 1e-6)

            key = f"{clip_id}_{int(s/sr*1000)}ms"
            np.savez_compressed(os.path.join(out_dir, key + ".npz"),
                                mel=mel.astype(np.float32),
                                path=path, start=float(s/sr), duration=float(window_s))

        processed += 1

        # si se pasa el 10% se borra el dir y se crea
        if processed in thresholds:
            part_idx = thresholds[processed]
            archive_current_chunk(part_idx, src_dir=out_dir, dst_dir=ARCH_DIR)
            clean_npz_dir(out_dir)  # borra los npz del chunk
            os.makedirs(out_dir, exist_ok=True)

    # ultimo resto por si acaso
    rem_files = sum(len(files) for _,_,files in os.walk(out_dir))
    if rem_files > 0:
        part_idx += 1
        archive_current_chunk(part_idx, src_dir=out_dir, dst_dir=ARCH_DIR)
        clean_npz_dir(out_dir)

    print("[+] Precomputación + checkpoints completados.")

## 3.4-Redes Neuronales Convolucionales

### 3.4.0-Configuración inicial

In [ ]:
# Parametros
DROPOUT = 0.4 # 0.3 #

### 3.4.1-Red Neuronal Convolucional simple

In [ ]:
class BirdCNN(nn.Module):
    def __init__(self, num_classes, dropout=0.3, in_channels=1):  # <— 1 canal (log-Mel)
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),  
            nn.ReLU(),
            nn.MaxPool2d((3, 2)),                                  
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),                                  
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))                           
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        if x.ndim == 3:                      
            x = x.unsqueeze(1) # añade canal
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

### 3.4.2-Efficient CNN

In [ ]:
class BirdEfficientNet(nn.Module):
    def __init__(self, num_classes, in_chans=1, pretrained=True, dropout=0.3):
        super().__init__()
        # timm gestiona in_chans y el head; añadimos dropout con un head propio si quieres
        self.backbone = timm.create_model(
            'efficientnet_b0',
            pretrained=pretrained,
            in_chans=in_chans,
            num_classes=0  # quitamos la head para añadir la nuestra con dropout
        )

        feat_dim = self.backbone.num_features
        
        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(feat_dim, num_classes)
        )

    def forward(self, x):
        if x.ndim == 3:
            x = x.unsqueeze(1)
        feats = self.backbone(x) # usa pesos pretrained
        logits = self.head(feats) # head aprende
        return logits

### 3.4.3-Resnet CNN

In [ ]:
class BirdResNet(nn.Module):
    def __init__(self, num_classes, in_chans=1, pretrained=False, dropout=0.3):
        super().__init__()
        if pretrained:
            try:
                base = tv.resnet18(weights=tv.ResNet18_Weights.IMAGENET1K_V1)
            except Exception:
                base = tv.resnet18(pretrained=True)
        else:
            try:
                base = tv.resnet18(weights=None)
            except Exception:
                base = tv.resnet18(pretrained=False)

        # conv1 adapt
        orig = base.conv1
        if in_chans != orig.in_channels:
            new_conv = nn.Conv2d(in_chans, orig.out_channels,
                                 kernel_size=orig.kernel_size, stride=orig.stride,
                                 padding=orig.padding, bias=False)
            if pretrained:
                with torch.no_grad():
                    w = orig.weight.data
                    if in_chans == 1:
                        new_conv.weight.copy_(w.mean(1, keepdim=True))
                    else:
                        new_conv.weight.copy_(w.mean(1, keepdim=True).repeat(1, in_chans, 1, 1))
            base.conv1 = new_conv

        # backbone = todo menos la fc
        in_features = base.fc.in_features
        base.fc = nn.Identity()
        self.backbone = base

        # head
        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        if x.ndim == 3:
            x = x.unsqueeze(1)
        feats = self.backbone(x)
        logits = self.head(feats)
        return logits

## 3.5-Distribuir dataset Entrenamiento/Validacion
Se verifica que haya al muestra de cada especie entre validacion y entrenamiento. La mayoria de especies tiene mas de 5 muestras, pero el problema esta en las que no. Por esos e utiliza StratifiedGroupKFold

In [ ]:
group_col = "filename"

if FLAG_LOAD_CSV:
    origen = "/kaggle/input/prep-mels"
    destino = path_csvs # se guarda en data/csv
    for file in os.listdir(origen):
        file = os.path.join(origen, file)
        if file.endswith('.csv'):
            shutil.copy(file, destino)
            print(f"[+] Copiado: {file}")

    # Cargar splits ya existentes, NO regenerarlos
    train = pd.read_csv(get_path(path_csvs, "index_train.csv"))
    val = pd.read_csv(get_path(path_csvs, "index_val.csv"))
    test_path = get_path(path_csvs, "index_test.csv")
    
    if os.path.exists(test_path):
        test = pd.read_csv(test_path)
        HAS_TEST = True
        print("[+] index_test.csv cargado")
    else:
        test = None
        HAS_TEST = False
        print("[INFO] No se ha encontrado index_test.csv")

else:    
    df = pd.read_csv(get_path(path_csvs, "index_audio.csv"))
    
    counts = df["primary_label"].value_counts()
    
    # Clases con menos de 2 muestras: no se pueden repartir bien
    # Se mandan a train
    rare = counts[counts < 3].index
    
    df_rare = df[df.primary_label.isin(rare)].copy()
    df_common = df[~df.primary_label.isin(rare)].copy()
    
    # ============================================================
    # Split inicial: TRAIN / VAL_BIG -> aprox 80 / 20
    # ============================================================
    
    sgkf = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
    
    train_idx, val_idx = next(
        sgkf.split(
            df_common,
            df_common["primary_label"],
            groups=df_common[group_col]
        )
    )
    
    train = df_common.iloc[train_idx].copy()
    val_big = df_common.iloc[val_idx].copy()
    
    # Clases con una sola muestra total van a train
    train = pd.concat([train, df_rare], ignore_index=True)
    
    
    # ============================================================
    # Si FLAG_MAKE_TEST = True: 
    #    VAL_BIG -> VAL / TEST
    #    n_splits=8 => test aprox - 12.5% de val_big
    #    STest tendra unos 700 audios aprox igual que inferencia de BirdClef
    # ============================================================
    if FLAG_MAKE_TEST:
        sgkf_val_test = StratifiedGroupKFold(
            n_splits=8,
            shuffle=True,
            random_state=42
        )
    
        val_final_idx, test_idx = next(
            sgkf_val_test.split(
                val_big,
                val_big["primary_label"],
                groups=val_big[group_col]
            )
        )
    
        val = val_big.iloc[val_final_idx].copy()
        test = val_big.iloc[test_idx].copy()
    
    else:
        val = val_big.copy()
        test = None
    
    # ============================================================
    # Post-fix: asegurar que train tiene todas las clases posibles
    # Nunca se toca test
    # ============================================================
    missing_in_train = set(df["primary_label"]) - set(train["primary_label"])
    
    for cls in list(missing_in_train):
        cand = val[val.primary_label == cls]
        if len(cand):
            take = cand.sample(1, random_state=42)
            train = pd.concat([train, take], ignore_index=True)
            val = val.drop(take.index)
    
    # ============================================================
    # Post-fix: si una clase tiene >=2 muestras, intentar que aparezca en val
    # Nunca se toca test
    # ============================================================
    for cls in set(df["primary_label"]):
        if cls not in set(val["primary_label"]) and counts[cls] >= 2:
            cand = train[train.primary_label == cls]
            if len(cand) > 1:
                take = cand.sample(1, random_state=42)
                val = pd.concat([val, take], ignore_index=True)
                train = train.drop(take.index)
    
    
    train = train.reset_index(drop=True)
    val = val.reset_index(drop=True)
    
    train.to_csv(get_path(path_csvs, "index_train.csv"), index=False)
    val.to_csv(get_path(path_csvs, "index_val.csv"), index=False)
    
    print("[+] Archivo index_train.csv guardado")
    print("[+] Archivo index_val.csv guardado")
    
    if FLAG_MAKE_TEST:
        test = test.reset_index(drop=True)
        test.to_csv(get_path(path_csvs, "index_test.csv"), index=False)
        print("[+] Archivo index_test.csv guardado")
        HAS_TEST = True
    else:
        test = None
        HAS_TEST = False

#####################################
# Verificaciones de los subconjuntos 
# train, val y test si aplica
####################################

assert set(train[group_col]).isdisjoint(set(val[group_col]))

if HAS_TEST:
    assert set(train[group_col]).isdisjoint(set(test[group_col]))
    assert set(val[group_col]).isdisjoint(set(test[group_col]))

# Si rep existe
assert set(train[group_col]).isdisjoint(set(rep[group_col]))
assert set(val[group_col]).isdisjoint(set(rep[group_col]))

if HAS_TEST:
    assert set(test[group_col]).isdisjoint(set(rep[group_col]))

print(f"[+] Nº clases en entrenamiento: {train['primary_label'].nunique()}")
print(f"[+] Nº clases en validación: {val['primary_label'].nunique()}")

if HAS_TEST:
    print(f"[+] Nº clases en test: {test['primary_label'].nunique()}")

print(f"[+] Nº muestras train: {len(train)}")
print(f"[+] Nº muestras val: {len(val)}")

if HAS_TEST:
    print(f"[+] Nº muestras test: {len(test)}")

total_split = len(train) + len(val) + (len(test) if HAS_TEST else 0)

print("[+] Porcentaje train:", round(len(train) / total_split, 4))
print("[+] Porcentaje val:", round(len(val) / total_split, 4))

if HAS_TEST:
    print("[+] Porcentaje test:", round(len(test) / total_split, 4))

print("[+] Solapamiento train-val:", len(set(train["filename"]) & set(val["filename"])))

if HAS_TEST:
    print("[+] Solapamiento train-test:", len(set(train["filename"]) & set(test["filename"])))
    print("[+] Solapamiento val-test:", len(set(val["filename"]) & set(test["filename"])))
    test_counts = test["primary_label"].value_counts()
    print("[+] Clases en test con 1 muestra:", (test_counts == 1).sum())
    print("[+] Clases en test con >=2 muestras:", (test_counts >= 2).sum())

## 3.6-Preprocesamiento de datos de validación
Ver apartado 2.1.0-Configuracion global

In [ ]:
if FLAG_PROCESS:
    #clean_folder("./data/val_npz")
    print("[*] Preprocesando datos de validacion")
    precompute_npz_chunked()
else:
    print("[+] Preprocesado de validacion desactivado")

# 4.-Entrenamiento

## 4.1-Configuracion

### 4.1.1-Funcion calcular pesos

In [ ]:
# Calcular pesos por clase
def calcular_pos_weight(train_csv_path: str, species_cols: list, device="cuda"):
    df = pd.read_csv(train_csv_path)
    conteo = Counter(df["primary_label"].dropna())
    total = sum(conteo.values())

    # Diccionario con la frecuencia de cada clase
    pesos = {cls: total / freq for cls, freq in conteo.items()}
    
    weights = [pesos.get(cls, 1.0) for cls in species_cols]

    # Crear tensor en CPU
    weights_tensor = torch.tensor(weights, dtype=torch.float32)

    # Ningún peso mayor que 20
    weights_tensor = torch.clamp(weights_tensor, max=20.0)
    
    # Normalizar vector para que su media sea 1
    weights_tensor = weights_tensor / weights_tensor.mean()

    print("[INFO] pos_weight stats:",
          "min", float(weights_tensor.min()),
          "mean", float(weights_tensor.mean()),
          "max", float(weights_tensor.max()))

    print("[INFO] clases >= 2:", int((weights_tensor >= 2).sum()))
    print("[INFO] clases >= 5:", int((weights_tensor >= 5).sum()))
    print("[INFO] clases >= 10:", int((weights_tensor >= 10).sum()))
    
    # Top 10 más grandes
    top = torch.topk(weights_tensor, k=10)
    print("[INFO] top10:", [
        (species_cols[i], float(w)) 
        for i, w in zip(top.indices.numpy(), top.values.numpy())
    ])

    # Pasar a GPU solo al final
    weights_tensor = weights_tensor.to(device)
    
    return weights_tensor

### 4.1.2-Configuracion

In [ ]:
# Activar o desactivar entrenamiento
FLAG_TRAIN = True #True si se quiere hacer un entrenamiento

# Configuracion del modelo
model_name= "resnet" #opciones: 'cnn', 'effnet', 'resnet'
NUM_CLASSES=206
use_pretrained = True # Solo afecta Effnet y Resnet

# Ruta a los csv
INDEX_VAL = get_path(path_csvs, "index_val.csv")
INDEX_TRAIN = get_path(path_csvs, "index_train.csv")

# Configuracion dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"

# Configuracion del entrenamiento
BATCH_SIZE = 16 
EPOCHS = 20 # spec augmentantion 18-25, sin 8-12
EARLY_STOP_PATIENCE = 5 #3 # nº epochas para que pare el entrenamiento si no mejora
APPLY_EARLY_STOP = True # True para activarlo

# Configuracion checkpoints y logs
clean_folder(path_models)
FLAG_CHECKPOINT = False # True si se quiere cargar un checkpoint externo
CHECKPOINT_NAME = model_name + "_ep004.pth" # nombre del checkpoint del modelo a cargar
CHECKPOINT_DIR = create_folder(path_models, "checkpoint")
csv_log = get_path(path_models, "metricas_log.csv")
model_prefix = model_name


# configuracion optimizador y critero
LR = 3e-4 #1e-4 #5e-4 #1e-3 # learning rate
LR_HEAD = 1e-4 # rango: 1e-4 <-> 1e-3 | lr para la cabeza effnet o resnet si usa pretrained
LR_BACKBONE = 1e-5 # rango: 1e-5 <-> 3e-5
FREEZE = True # True si se quiere aplicar freeze a backbone alguna epoch(Solo effnet y resent con pretrained)
EPOCHS_FROZEN = 2 # nº epochs congeladas backbone, 1-2 suficiente
weights_tensor = calcular_pos_weight(INDEX_TRAIN, valid_labels, device)
WEIGHT_DECAY = 3e-4 # 1e-5 cnn # Regularizacion L2, rango [1e-6, 1e-4], 1e-4 muy fuerte, penzaliza mucho
LOSS_FN = torch.nn.BCEWithLogitsLoss(pos_weight=weights_tensor) # criterion 

# Configuracion SCHEDULER (Reduce LR on Plateau)
use_scheduler = True
scheduler_mode = "max"  # 'max' si usas AUC, 'min' si usas loss
scheduler_patience = 3
scheduler_factor = 0.5
scheduler_min_lr = 1e-6 # valor minimo de lr

# Configuracion Pooling, calcular probabilidades de predicciones
POOL = "topk" # valores: mean,max,mix,topk
ALPHA = 0.3 # 0.5 # Para pool mix, 0.5 muy fuerte para max
TOPK = 3 # N ventans sobre las que saca topk

# Archivos npz
NPZ_PATH = "/kaggle/input/prep-mels/val_npz/val_npz/"

# Configuracion del sistema
NUM_WORKERS = 0
SHUFFLE = True
PIN_MEMORY = (device == "cuda")

### 4.1.3-Creacion de los Data Loaders

In [ ]:
def parse_clip_and_ms(base: str):
    stem = base[:-4]
    left, ms_part = stem.rsplit("_", 1)
    ms = int(ms_part.replace("ms", ""))
    return left, ms

dfv = pd.read_csv(INDEX_VAL)
dfv["filename"] = dfv["filename"].astype(str).str.strip()

# si no existe 'path', la construimos
if "path" not in dfv.columns:
    dfv["path"] = dfv["filename"].apply(lambda fn: os.path.join(path_train_audio, fn))

dfv["path"] = dfv["path"].astype(str).str.strip()
dfv["clip_id"] = dfv["filename"].str.replace(".ogg","", regex=False).str.replace("/", "__", regex=False)
clip2path = dict(zip(dfv["clip_id"], dfv["path"]))

print("[INFO] clip2path size:", len(clip2path))

In [ ]:
# Creacion DataLoaders

train_dataset = OnTheFlyWindows(index_csv=INDEX_TRAIN,label_map=label_to_index, split="train",
                                bandpass=APPLY_BANDPASS , specaug=APPLY_SPECAUGMENT,
                               time_mask=TIME_MASK, freq_mask=FREQ_MASK, prob=PROB)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=SHUFFLE, num_workers=NUM_WORKERS,
                          pin_memory=PIN_MEMORY)

files = sorted(gb.glob(NPZ_PATH + "*.npz"))
print("[+] NPZ files:", len(files))


groups = collections.defaultdict(list)
miss = 0

for f in files:
    base = os.path.basename(f)
    try:
        clip_id, ms = parse_clip_and_ms(base)
    except Exception:
        miss += 1
        continue

    audio_path = clip2path.get(clip_id)   # ahora debería existir
    if audio_path is None:
        miss += 1
        continue

    groups[audio_path].append(f)

print("[INFO] groups:", len(groups), "miss:", miss)

base = os.path.basename(files[0])
print(f"[INFO] Test primero archivo base: {base}")

val_dataset_npz = ValGroupedDataset(groups, INDEX_VAL, label_to_index, num_classes=206)

print("len(val_dataset_npz)=", len(val_dataset_npz))
print("len(train_dataset)=", len(train_dataset))

# Detectar nº de canales de entrada
batch = next(iter(train_loader))          # xb: [B, C, MELS, T] o [B, MELS, T]
if isinstance(batch, (list, tuple)):
    xb = batch[0]             # primer elemento = inputs
elif isinstance(batch, dict):
    xb = batch.get('inputs', next(iter(batch.values())))
else:
    xb = batch                # por si viniera directo

in_chans = 1 if xb.ndim == 3 else xb.shape[1]
print('[INFO] FORMATO: xb shape:', tuple(xb.shape), 'in_chans:', in_chans)

**Pequeña prueba** para comprobar si se separa bien:

### 4.1.4-Creacion del Modelo

In [ ]:
# Crear modelo
if model_name=="cnn":
    model = BirdCNN(num_classes=NUM_CLASSES)
elif model_name=="effnet":
    model = BirdEfficientNet(num_classes=NUM_CLASSES, in_chans=in_chans, pretrained=use_pretrained)
elif model_name=="resnet":
    model = BirdResNet(num_classes=NUM_CLASSES, in_chans=in_chans, pretrained=use_pretrained)
else:
    raise ValueError("[X] El nombre del modelo es: cnn, efnet, resnet")

# Enviar al dispositivo, gpu o cpu
model = model.to(device)

print(f"[+] Cargando modelo {model_name}")
print(f"[+] Utilizando dispositivo {device}")

## 4.2-Clase BirdTrainer

### 4.2.1- Funciones para aplicar FREEZE BACKBONE

In [ ]:
def get_scheduler(optimizer):
    """
    Crea un scheduler para el optimizador usando la configuración global actual.
    """
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode=scheduler_mode,
        patience=scheduler_patience,
        factor=scheduler_factor,
        min_lr=scheduler_min_lr,
    )

def set_backbone_bn_eval(model):
    """
    Pone en eval() las BatchNorm del backbone para que NO actualicen running_mean/var
    mientras el backbone esté congelado.
    """
    if not hasattr(model, "backbone"):
        return
    for m in model.backbone.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d, nn.SyncBatchNorm)):
            m.eval()

def freeze_backbone_(model, freeze=True):
    if hasattr(model, "backbone"):
        for p in model.backbone.parameters():
            p.requires_grad = not freeze

def make_optimizer(model, model_name, LR, WEIGHT_DECAY,
                   lr_backbone=None, lr_head=None,
                   use_pretrained=False, freeze_backbone=False):

    # Optimizador otra vez
    Optim = torch.optim.AdamW

    model_name = model_name.lower()

    # CNN: todo junto
    if model_name == "cnn":
        return Optim(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # EffNet/ResNet pretrained: 2 LRs
    if use_pretrained and model_name in ["effnet", "resnet"]:
        # Freeze si toca
        freeze_backbone_(model, freeze_backbone)

        # Si backbone congelado: optimiza SOLO head
        if freeze_backbone:
            if hasattr(model, "head"):
                params = model.head.parameters()
            else:
                # fallback por si no existe head
                params = [p for p in model.parameters() if p.requires_grad]
            return Optim(params, lr=lr_head or LR, weight_decay=WEIGHT_DECAY)

        # Unfreeze: param groups (backbone/head)
        if hasattr(model, "backbone") and hasattr(model, "head"):
            return Optim([
                {"params": model.backbone.parameters(), "lr": lr_backbone or (LR * 0.1)},
                {"params": model.head.parameters(),     "lr": lr_head or LR},
            ], weight_decay=WEIGHT_DECAY)

        # Fallback generico (si tu ResNet no tiene backbone/head), timm llama a classifier/fc/head
        head_params = []
        backbone_params = []
        for n, p in model.named_parameters():
            if any(k in n.lower() for k in ["classifier", "fc", "head"]):
                head_params.append(p)
            else:
                backbone_params.append(p)

        return Optim([
            {"params": backbone_params, "lr": lr_backbone or (LR * 0.1)},
            {"params": head_params,     "lr": lr_head or LR},
        ], weight_decay=WEIGHT_DECAY)

    # EffNet/ResNet sin pretrained o sin split: normal
    return Optim(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

In [ ]:
class BirdTrainer:
    def __init__(self, model, optimizer, criterion, scheduler=None, device="cuda", 
                 early_stop_patience=3, checkpoint_dir="checkpoints", apply_early_stop=False,
                 csv_log="metrics_log.csv", model_prefix="model", amp=True,
                 pool="mix", alpha=0.5, topk=3, 
                 use_pretrained=False,freeze=False, epochs_frozen=1):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.criterion = criterion
        self.scheduler = scheduler
        self.device = device
        self.amp = amp and (device == "cuda")
        self.scaler = torch.cuda.amp.GradScaler(enabled=self.amp)
        
        self.checkpoint_dir = Path(checkpoint_dir)
        self.csv_log = csv_log
        self.model_prefix = model_prefix

        # Early stopping
        self.early_stop_patience = early_stop_patience
        self.apply_early_stop = apply_early_stop
        self.epochs_since_impr = 0

        # Pooling config
        self.alpha = alpha
        self.topk = topk
        allowed = {"mean", "max", "mix", "topk"}
        if pool not in allowed:
            self.pool = "topk"
            print(f"[WARNING] No esta bien asigando el pooling, valor: {pool}. Se asigna topk pooling.")
        else:
            self.pool = pool
            print(f"[INFO] Pooling utillizado: {pool}")

        # Freeze Backbone
        self.use_pretrained = use_pretrained
        self.freeze = freeze
        self.epochs_frozen = epochs_frozen
        if self.freeze and self.use_pretrained and (self.model_prefix in ["resnet", "effnet"]):
            print("[+] Utilizando Freeze Backbone con pretrained")
        else:
            self.freeze = False
            self.use_pretrained = False
        
        # checkpoints
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.path_last = self.checkpoint_dir / f"{model_prefix}_last.pth"
        self.path_best = self.checkpoint_dir / f"{model_prefix}_best.pth"
        
        self.epoch_global = self._load_checkpoint(self.path_last)
        # actualizar optimizador si es necesario
        self._refresh_optim_if_needed(self.epoch_global) 
        if getattr(self, "_ckpt_pending_optim", None) is not None:
            try:
                self.optimizer.load_state_dict(self._ckpt_pending_optim)
                print("[+] optim_state cargado correctamente")
            except ValueError as e:
                print(f"[!] optim_state NO compatible ({e}). Se reinicia optimizador.")
        self._ckpt_pending_optim = None
        
        self.best_auc = -1.0
        if self.path_best.is_file():
            self.best_auc = torch.load(self.path_best, map_location="cpu", weights_only=False)["metrics"]["val_macro_auc"]
            print(f"[+] Mejor AUC previo encontrado: {self.best_auc:.4f}")

    def _save_checkpoint(self, epoch: int, metrics: dict, path: Path):
        torch.save({
            "epoch": epoch,
            "model_state": self.model.state_dict(),
            "optim_state": self.optimizer.state_dict(),
            "metrics": metrics,
        }, path)

    def _load_checkpoint(self, path: Path) -> int:
        if path.is_file():
            ckpt = torch.load(path, map_location="cpu", weights_only=False)
            self.model.load_state_dict(ckpt["model_state"])
            print(f"[+] Checkpoint cargado (época {ckpt['epoch']+1})")
            self._ckpt_pending_optim = ckpt.get("optim_state", None)  # guardamos para intentar luego
            return ckpt["epoch"] + 1
        print(f"[+] No hay checkpoint cargado, se empieza desde 0.")
        self._ckpt_pending_optim = None
        return 0

    def _append_metrics_csv(self, row_dict: dict):
        file_exists = Path(self.csv_log).is_file()
        with open(self.csv_log, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=row_dict.keys())
            if not file_exists:
                writer.writeheader()
            writer.writerow(row_dict)

    # metricas
    def _macro_auc_skip_empty(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        per_class_auc = [roc_auc_score(y_true[:, k], y_pred[:, k])
                         for k in range(y_true.shape[1])
                         if y_true[:, k].sum() > 0 and np.unique(y_true[:, k]).size > 1]
        return float(np.mean(per_class_auc)) if per_class_auc else 0.0

    def _compute_metrics(self, y_true, y_pred):
        y_pred_bin = (y_pred > 0.5).astype(int)
        macro_auc = self._macro_auc_skip_empty(y_true, y_pred)
        f1_macro = f1_score(y_true, y_pred_bin, average="macro", zero_division=0)
        f1_micro = f1_score(y_true, y_pred_bin, average="micro", zero_division=0)
        return macro_auc, f1_macro, f1_micro

    def _refresh_optim_if_needed(self, epoch: int):
        """
        Crea/Recrea optimizer (+ scheduler) SOLO cuando cambia freeze_now.
        Devuelve freeze_now para que se pueda usar si se necesita.
        Freeze/unfreeze se decide usando epoch_global, así que al reanudar 
        desde checkpoint se respeta la fase global real del entrenamiento.
        No se reinicia el freeze en cada llamada a train(): si ya se aplicó antes,
        al cargar el checkpoint no vuelve a congelar.
        
        """
        if not (self.freeze and self.use_pretrained and self.model_prefix in ["effnet", "resnet"]):
            return False

        # verifica si debe estar congelado backbone todavia
        freeze_now = (epoch < self.epochs_frozen)

        # Inicializa atributo si no existe
        if not hasattr(self, "_prev_freeze_state"):
            self._prev_freeze_state = None

        # Si cambia el estado: recrea optimizer y scheduler
        if freeze_now != self._prev_freeze_state:
            self.optimizer = make_optimizer(
                model=self.model,
                model_name=self.model_prefix,
                LR=LR,
                WEIGHT_DECAY=WEIGHT_DECAY,
                lr_backbone=LR_BACKBONE,
                lr_head=LR_HEAD,
                use_pretrained=self.use_pretrained,
                freeze_backbone=freeze_now
            )

            self.scheduler = get_scheduler(self.optimizer)

            self._prev_freeze_state = freeze_now
            
        # Si está congelado, fija BN del backbone
        if freeze_now:
            set_backbone_bn_eval(self.model)

        return freeze_now

    # validacion con pooling por audio (agrupa por 'path')
    def _eval_grouped(self, val_loader, eval_bs=64):
        self.model.eval()
        y_true_list, y_pred_list = [], []
    
        def to_numpy_onehot(y):
            if isinstance(y, torch.Tensor):
                y = y.detach().cpu()
            return (y.squeeze(0).numpy() if getattr(y, "ndim", 1) == 2 else y.numpy())
    
        with torch.inference_mode():
            for item in tqdm(val_loader, desc="[Validacion]"):
                # Desempaqueta: puede venir (file_list, y, base) o (f, y, base)
                f_part, y_onehot, base = item
    
                # Normaliza y_onehot
                y_true = to_numpy_onehot(y_onehot)
    
                # ---- Normaliza a lista de rutas ----
                # case A: batch_size=1 y vienen listas/tuplas
                if isinstance(f_part, (list, tuple)):
                    file_list = f_part[0] if (len(f_part) == 1 and isinstance(f_part[0], (list, tuple))) else f_part
                else:
                    file_list = f_part  # podría ser str o tensor
    
                # Si es str/tensor único, conviértelo a lista
                if isinstance(file_list, (str, bytes)) or isinstance(file_list, torch.Tensor):
                    file_list = [file_list]
    
                # Asegura que cada elemento es str
                norm_paths = []
                for f in file_list:
                    if isinstance(f, torch.Tensor):
                        f = f.item() if f.ndim == 0 else f[0]
                    norm_paths.append(str(f))
                file_list = norm_paths
    
                # ---- Carga todas las ventanas de ese audio ----
                mels = []
                for f in file_list:
                    with np.load(f) as z:
                        mel = z["mel"] if "mel" in z else z["mels"][0]
                        mels.append(mel)
                mels = np.stack(mels)  # [N, 128, T]
    
                # ---- Inferencia por micro-batches + pooling ----
                preds_win = []
                for i in range(0, len(mels), eval_bs):
                    xb = torch.tensor(mels[i:i+eval_bs], dtype=torch.float32).unsqueeze(1).to(self.device)  # [b,1,128,T]
                    logits = self.model(xb)
                    preds_win.append(torch.sigmoid(logits).cpu().numpy())
                #p_audio = np.vstack(preds_win).mean(axis=0)  # pooling: media
                #p_audio = 0.5 * preds.mean(axis=0) + 0.5 * preds.max(axis=0)
                preds = np.vstack(preds_win)

                n = preds.shape[0] # nº ventanas

                if self.pool == "mean":
                    p_audio = preds.mean(axis=0)
                elif self.pool == "max":
                    p_audio = preds.max(axis=0)
                elif self.pool == "mix":
                    p_audio = (1 - self.alpha) * preds.mean(axis=0) + self.alpha * preds.max(axis=0)
                else:  # "topk" (self.topk = 3 por defecto)
                    if n <= self.topk:
                        # No hay suficientes ventanas para topk -> usa mix(mean, max)
                        p_audio = (1 - self.alpha) * preds.mean(axis=0) + self.alpha * preds.max(axis=0)
                    else:
                        # Topk real (si n=3 usa 3, si n=4 sigue usando 3 mejores)
                        #k = min(self.topk, preds.shape[0])
                        #topk_mean = np.sort(preds, axis=0)[-k:, :].mean(axis=0)
                        topk_mean = np.sort(preds, axis=0)[-self.topk:, :].mean(axis=0)
                        mean = preds.mean(axis=0)
                        p_audio = 0.5 * mean + 0.5 * topk_mean

                y_true_list.append(y_true)
                y_pred_list.append(p_audio)
    
        return np.vstack(y_true_list), np.vstack(y_pred_list)



    # entrenamiento: loaders externos
    def train_from_dataloaders(self, train_loader: DataLoader, val_loader: DataLoader, num_epochs: int):
        print(f"[*] Entrenando con {len(train_loader.dataset)} muestras...")

        # estado anterior de freeze backbone si es necesario
        
        for epoch in range(self.epoch_global, self.epoch_global + num_epochs):
            print(f"\n[*] Época {epoch+1} iniciada")

            # si Freeze sta activado, actualiza el optimizador si es necesario(1 vez al menos)
            freeze_now = self._refresh_optim_if_needed(epoch)
                

            # --------- TRAIN ---------
            self.model.train()
            running_loss = 0.0
            num_batches = len(train_loader)
            print(f"[*] Entrenando: {num_batches} batches")
            for batch in tqdm(train_loader, desc=f"[Train Epoch {epoch+1}]"):
                # soporta (x,y), (x,y,path) o dict {"mel","label",...}
                if isinstance(batch, dict):
                    xb, yb = batch["mel"], batch["label"]
                elif isinstance(batch, (list, tuple)):
                    xb, yb = batch[0], batch[1]   # ignora path si viene en batch[2]
                else:
                    xb, yb = batch

                xb = xb.float()
                yb = yb.float()  
                xb = xb.to(self.device)
                yb = yb.to(self.device)
                self.optimizer.zero_grad()

                if yb.ndim == 1:  # [B] -> [B,206]
                    y_oh = torch.zeros(yb.size(0), 206, device=yb.device)
                    y_oh[torch.arange(yb.size(0)), yb.long()] = 1.0
                    yb = y_oh

                if self.amp:
                    with torch.cuda.amp.autocast():
                        logits = self.model(xb)
                        loss = self.criterion(logits, yb)
                    self.scaler.scale(loss).backward()
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                else:
                    logits = self.model(xb)
                    loss = self.criterion(logits, yb)
                    loss.backward()
                    self.optimizer.step()

                running_loss += loss.item() * xb.size(0)

            train_loss = running_loss / len(train_loader.dataset)
            print(f"[+] Entrenamiento completado — Loss: {train_loss:.4f}")

            # --------- VALID ---------
            print(f"[*] Validando con {len(val_loader)} muestras...")
            #y_true, y_pred = self._eval_grouped(val_loader)  # pooling por audio
            y_true, y_pred = self._eval_grouped(val_loader, eval_bs=64)
            print(f"[INFO] Pred min/mean/max:", y_pred.min(), y_pred.mean(), y_pred.max())
            print("[INFO] clases en val:", int(y_true.sum(axis=0).astype(bool).sum()))
            i = 0
            p = y_pred[i]
            top = p.argsort()[-5:][::-1]
            print("[INFO] top5:", [(CLASSES[j], float(p[j])) for j in top])
            print("[INFO] true:", [CLASSES[j] for j in np.where(y_true[i]>0)[0]])

            macro_auc, f1_macro, f1_micro = self._compute_metrics(y_true, y_pred)

            if self.scheduler:
                # usa 'max' si monitorizas AUC
                self.scheduler.step(macro_auc)
                print(f"[+] Scheduler actualizado — nuevo LR: {self.optimizer.param_groups[0]['lr']:.6f}")
                pg = self.optimizer.param_groups[0]
                print(f"[+] Weight decay = {pg.get('weight_decay', 0)}")

            param_groups = self.optimizer.param_groups

            log_row = {
                "epoch": epoch + 1,
                "train_loss": round(train_loss, 6),
                "val_macro_auc": round(macro_auc, 6),
                "val_f1_macro": round(f1_macro, 6),
                "val_f1_micro": round(f1_micro, 6),
                "model_name": self.model_prefix,
                "use_pretrained": self.use_pretrained,
                "freeze_backbone": freeze_now,
                "num_param_groups": len(param_groups),
            }
            
            if self.model_prefix == "cnn":
                log_row["learning_rate"] = round(param_groups[0]["lr"], 10)
                log_row["lr_backbone"] = None
                log_row["lr_head"] = None
            
            elif len(param_groups) == 1:
                log_row["learning_rate"] = None
                log_row["lr_backbone"] = None
                log_row["lr_head"] = round(param_groups[0]["lr"], 10)
            
            else:
                log_row["learning_rate"] = None
                log_row["lr_backbone"] = round(param_groups[0]["lr"], 10)
                log_row["lr_head"] = round(param_groups[1]["lr"], 10)

            #guardar en csv
            self._append_metrics_csv(log_row)

            print(f"[Ep {(epoch+1):03d}] Loss: {train_loss:.4f} | AUC: {macro_auc:.4f} | F1-ma: {f1_macro:.4f} | F1-mi: {f1_micro:.4f}")

            # guarda la ultima epoca entrenada
            self._save_checkpoint(epoch, log_row, self.path_last)
            print(f"[+] Checkpoint guardado: {self.path_last}")

            # guarda cada epoca entrenada
            path_all = self.checkpoint_dir / f"{self.model_prefix}_ep{(epoch+1):03d}.pth"
            self._save_checkpoint(epoch, log_row, path_all)
            print(f"[+] Checkpoint EP:   {path_all}")

            # early stopping / best
            if macro_auc > self.best_auc + 1e-6:
                self.best_auc = macro_auc
                self.epochs_since_impr = 0
                self._save_checkpoint(epoch, log_row, self.path_best)
                print(f"[+] Nuevo BEST AUC {self.best_auc:.4f} — checkpoint actualizado: {self.path_best}")
            elif self.apply_early_stop:
                self.epochs_since_impr += 1
                print(f"[+] Sin mejora en validación (épocas sin impr.: {self.epochs_since_impr}/{self.early_stop_patience})")
                if self.epochs_since_impr >= self.early_stop_patience:
                    print(f"[!WARNING] Early stopping activado. Deteniendo entrenamiento.")
                    return

            self.epoch_global += 1
            torch.cuda.empty_cache(); gc.collect()

        print(f"\n[+] Entrenamiento finalizado. Último AUC: {macro_auc:.4f}")
        print(f"[+] Métricas en {self.csv_log}")
        print(f"[+] Último checkpoint en {self.path_last}")

## 4.3-Ultimas pruebas antes de entrenar
Comprueba si hay fuga de datos o colision de datos, y si estan bien las rutas de los archivos npz

In [ ]:
tr = pd.read_csv(INDEX_TRAIN)
va = pd.read_csv(INDEX_VAL)

inter = set(tr["filename"]) & set(va["filename"])
print("[+] INTERSECCIÓN train-val por filename:", len(inter))

if "path" in tr.columns and "path" in va.columns:
    inter_path = set(tr["path"]) & set(va["path"])
    print("[+] INTERSECCIÓN train-val por path:", len(inter_path))

## 4.4-Checkpoint, Optimizador, Scheduler y Trainer
Si FLAG de checkpoint esta activado, copia al output de kaggle los checkpoints. Tambien crea el optimizador, el scheduler y el trainer.

In [ ]:
print("[+] Modelo:", model_name)

if FLAG_CHECKPOINT:
    print(f"[*] Cargando checkpoint...")
    origen = "/kaggle/input/model-epochs/" + CHECKPOINT_NAME
    destino = CHECKPOINT_DIR + "/" + model_name + "_last.pth"
    shutil.copy(origen, destino)
    print(f"[+] Copiado: {origen} en {destino}")
    origen = "/kaggle/input/model-epochs/" + model_name + "_best.pth"
    destino = CHECKPOINT_DIR + "/" + model_name + "_best.pth"
    shutil.copy(origen, destino)
    print(f"[+] Copiado: {origen} en {destino}")
    origen = "/kaggle/input/model-epochs/" + "metricas_log.csv"
    if os.path.exists(origen):
        destino = csv_log
        shutil.copy(origen, destino)
        print(f"[+] Copiado: {origen} en {destino}")
    else:
        print("[WARNING] No hay csv de metricas en el input para cargar")
else:
    print("[+] Cargar checkpoint desactivado")

if use_pretrained and (model_name in ["effnet", "resnet"]):
    optimizer = make_optimizer(
        model=model,
        model_name=model_name,
        LR=LR,
        WEIGHT_DECAY=WEIGHT_DECAY,
        lr_backbone=LR_BACKBONE,
        lr_head=LR_HEAD,
        use_pretrained=True,
        freeze_backbone=FREEZE     # True en época 1 (o 2), luego False
    )
elif model_name == "cnn":
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
else: # futuro para panns
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode=scheduler_mode, patience=scheduler_patience,
    factor=scheduler_factor, min_lr=scheduler_min_lr
)


trainer = BirdTrainer(
    model=model,
    optimizer=optimizer,
    criterion=LOSS_FN,
    scheduler=scheduler,
    early_stop_patience=EARLY_STOP_PATIENCE,
    apply_early_stop=APPLY_EARLY_STOP,
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    csv_log=csv_log,
    model_prefix=model_prefix,
    amp=True,
    pool=POOL,
    alpha=ALPHA,
    topk=TOPK,
    freeze=FREEZE, 
    epochs_frozen=EPOCHS_FROZEN,
    use_pretrained=use_pretrained
)

## 4.5-Entrenamiento

In [ ]:
print("[+] Torch:", torch.__version__)
print("[+] CUDA torch:", torch.version.cuda)
print("[+] CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("[+] GPU:", torch.cuda.get_device_name(0))
    print("[+] Capability:", torch.cuda.get_device_capability(0))

In [ ]:
# --- Entrenamiento principal ---
if FLAG_TRAIN:
    trainer.train_from_dataloaders(train_loader, val_dataset_npz, num_epochs=EPOCHS)
else:
    print("[+] Train desactivado")

## 4.4-Ver rendimiento macro AUC/Epoch

In [ ]:
def visualizar(path_csv, save=True, path="./", name="val_macro_auc"):
    df = pd.read_csv(path_csv)
    epochs = df["epoch"]
    metricas  = df["val_macro_auc"]

    # pintar grafica
    plt.figure(figsize=(6, 4))
    plt.plot(epochs, metricas, marker="o") # sin especificar colores usa los de Matplotlib
    plt.xlabel("Epoch")
    plt.ylabel("Validacion Macro AUC")
    plt.title("Rendimiento Macro AUC por Epochs")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()

    if save:
        plot_path = os.path.join(path, f"{name}.png")
        plt.savefig(plot_path, dpi=300)   # 300 dpi = buena resolucion para informes
    # mostrar
    plt.show()

In [ ]:
if FLAG_TRAIN:
    visualizar(csv_log, path=path_models)
else:
    print("[+] Train desactivado")